# Bài 11 · Xử lý dữ liệu phi cấu trúc bằng LLM

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

Notebook bám theo slide Bài 11. Chạy tuần tự từ trên xuống.

> 💡 **Trước khi sửa:** File → *Save a copy in Drive* để lưu bản của riêng bạn.

**Mục tiêu bài học** — sau bài này, bạn:

1. Giải thích được vì sao dữ liệu văn bản tự do phải **đổi dạng thành bảng** trước khi phân tích.
2. Xây được **phương pháp đối chứng (baseline) không dùng LLM** bằng từ khoá hoặc từ điển.
3. Gọi được **Gemini API** từ Python với **đầu ra có cấu trúc** (schema + enum), có giới hạn tốc độ và cache.
4. **Đánh giá** chất lượng LLM so với phương pháp đối chứng trên bộ nhãn gán tay, đồng thời ước tính **chi phí** bằng token.

## 0. Chuẩn bị

### Lấy khóa API (miễn phí, không cần thẻ)

1. Vào [Google AI Studio](https://aistudio.google.com) → đăng nhập Google → **Get API key** → tạo key.
2. Trong Colab: bấm biểu tượng **🔑 Secrets** ở cột trái → **Add new secret** → Name: `GEMINI_API_KEY`, Value: dán key → bật **Notebook access**.

### Nếu không có khóa API

Bạn vẫn có thể học đầy đủ: notebook đã lưu sẵn **kết quả LLM trong cache** cho dữ liệu minh hoạ (`DEMO_MODE`).
Mọi cell đều chạy; kết quả LLM được đọc từ cache thay vì gọi API thật.
Đây cũng là cơ chế cache mà bài tập lớn yêu cầu qua cờ `--skip-llm`.

In [ ]:
%pip install -q google-genai pydantic

import json, re, time, os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Đọc khóa API: Colab Secrets -> biến môi trường -> không có (DEMO_MODE)
API_KEY = None
try:
    from google.colab import userdata          # chỉ có trên Colab
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

HAS_KEY = bool(API_KEY)
MODEL = "gemini-3.1-flash-lite"   # hạn mức miễn phí cao; thử "gemini-3.5-flash" với văn bản khó

if HAS_KEY:
    from google import genai
    client = genai.Client(api_key=API_KEY)
    print(f"🔑 Có khóa API — sẽ gọi Gemini thật (mô hình: {MODEL})")
else:
    print("ℹ️ Không có khóa API — DEMO_MODE dùng kết quả LLM trong cache. Notebook vẫn chạy đầy đủ.")

## 1. Dữ liệu minh hoạ: 24 đánh giá về chỗ ở

Tập dữ liệu gồm 24 đánh giá rút gọn theo kiểu Inside Airbnb, được viết bằng nhiều ngôn ngữ.
Quy mô này đủ nhỏ để bạn **đọc toàn bộ**, điều không thể làm với 690.000 đánh giá thật. Bài này tập trung xử lý cột `comments`.

In [ ]:
REVIEWS = [
    (1,  "Great location, five minutes from the metro. The host was lovely!"),
    (2,  "The flat was not clean at all, dust everywhere. Location was perfect though."),
    (3,  "Súper cómodo y muy limpio. El anfitrión respondió al instante."),
    (4,  "Ruidoso hasta las 3am, imposible dormir. No lo recomiendo."),
    (5,  "Apartamento bem localizado, mas o wifi caiu o tempo todo."),
    (6,  "Nothing special, but it was cheap and did the job."),
    (7,  "L'appartement est charmant mais la rue est très bruyante la nuit."),
    (8,  "Amazing value for money, spotless bathroom, super quiet neighborhood."),
    (9,  "Todo perfecto! Volveremos seguro."),
    (10, "The pictures are misleading — the room is tiny and smells of smoke."),
    (11, "Host cancelled last minute, we were stranded. Avoid!"),
    (12, "Muito limpo, cama confortável, anfitriã atenciosa. Recomendo!"),
    (13, "Decent place. A bit far from the center but the bus stop is close."),
    (14, "Perfect for a weekend getaway."),
    (15, "El barrio se siente inseguro de noche; el depto está bien."),
    (16, "Ottima posizione, appartamento pulito. Torneremo!"),
    (17, "Quiet street, comfy bed, kind host — everything was great."),
    (18, "We found hair in the sheets and the kitchen was greasy."),
    (19, "So loud! The bar downstairs plays music till 2am."),
    (20, "Un poco caro para lo que ofrece, pero la ubicación es inmejorable."),
    (21, "The check-in instructions were confusing; host took hours to reply."),
    (22, "Cozy studio, fast wifi, walkable to everything."),
    (23, "👍👍👍"),
    (24, ""),
]
df = pd.DataFrame(REVIEWS, columns=["review_id", "comments"]).set_index("review_id")
print(f"{len(df)} đánh giá; độ dài trung bình {df.comments.str.len().mean():.0f} ký tự")
df.head(6)

Câu hỏi ta muốn trả lời:

> **Khách chê điều gì nhiều nhất? Cảm xúc chung ra sao? Các đánh giá được viết bằng những ngôn ngữ nào?**

Muốn `groupby` được thì trước hết phải **tạo ra các cột** `sentiment`, `aspects`, `language` — chúng
chưa tồn tại.

## 2. Phương pháp đối chứng không dùng LLM

Xây dựng **phương pháp đối chứng trước, LLM sau**. Phương pháp đối chứng tạo ra mốc so sánh
để xác định mức cải thiện của LLM có tương xứng với chi phí và thời gian hay không.

In [ ]:
# 2a. Khía cạnh (aspect) bằng từ khoá — mỗi khía cạnh một pattern regex
# (bắt đầu thực tế: soạn bằng tiếng Anh — ngôn ngữ mình đọc được)
ASPECT_KEYWORDS = {
    "location":    r"location|metro|center|walkable",
    "cleanliness": r"clean|dust|dirty|hair|greasy|smell",
    "host":        r"host|check-in",
    "value":       r"value|price|cheap",
    "noise":       r"noisy|noise|loud|quiet",
    "amenities":   r"wifi|bed|kitchen|bathroom",
}

def baseline_aspects(text: str) -> list[str]:
    return [a for a, pat in ASPECT_KEYWORDS.items()
            if re.search(pat, text, flags=re.I)]

df["base_aspects"] = df["comments"].map(baseline_aspects)
df[["comments", "base_aspects"]].head(8)

In [ ]:
# 2b. Trích xuất cảm xúc bằng từ điển: đếm từ tích cực trừ từ tiêu cực (cũng tiếng Anh)
POS_WORDS = r"great|perfect|amazing|excellent|lovely|comfy|cozy|spotless|kind|fast|quiet"
NEG_WORDS = r"not |dirty|noisy|loud|avoid|cancelled|misleading|smell|hair|greasy|confusing|stranded|tiny"

def baseline_sentiment(text: str) -> str:
    pos = len(re.findall(POS_WORDS, text, flags=re.I))
    neg = len(re.findall(NEG_WORDS, text, flags=re.I))
    return "positive" if pos > neg else "negative" if neg > pos else "mixed"

df["base_sent"] = df["comments"].map(baseline_sentiment)
df["base_sent"].value_counts()

In [ ]:
# 2c. Trích xuất ngôn ngữ bằng "stopword đặc trưng" — heuristic 10 dòng
LANG_HINTS = {
    "en": r"\b(the|was|and|but|from)\b",
    "es": r"\b(el|la|muy|pero|hasta|todo)\b",
    "pt": r"\b(o|muito|mas|tempo|bem)\b",
    "fr": r"\b(le|la|est|très|mais)\b",
    "it": r"\b(ottima|molto|appartamento)\b",
}

def baseline_lang(text: str) -> str:
    counts = {lang: len(re.findall(pat, text, flags=re.I))
              for lang, pat in LANG_HINTS.items()}
    best = max(counts, key=counts.get)
    return best if counts[best] > 0 else "und"   # und = không xác định được ngôn ngữ

df["base_lang"] = df["comments"].map(baseline_lang)
df[["comments", "base_sent", "base_lang"]].tail(6)

Kết quả trên cho thấy một số hạn chế quen thuộc:

- Đánh giá 2 *"not clean at all"* chứa từ khoá `clean`; phương pháp đối chứng không hiểu phủ định.
- Các đánh giá bằng tiếng Tây Ban Nha, Bồ Đào Nha và Ý hầu như **không được nhận diện** bởi bộ từ khoá tiếng Anh;
  muốn phủ thêm một ngôn ngữ là phải soạn lại từ điển cho ngôn ngữ đó.
- Đánh giá 23 chỉ có emoji và đánh giá 24 rỗng nên phương pháp đối chứng chỉ có thể trả `und`/`mixed`.

Chưa thể kết luận chỉ từ quan sát này. Mục 5 sẽ đánh giá hai phương pháp bằng số liệu.

## 3. Gọi LLM từ Python

Trong pipeline, mã nguồn gửi **yêu cầu**, nhận **phản hồi** và có thể lặp lại hàng nghìn lần.
Hãy thử lời gọi đầu tiên; cell này chỉ chạy khi có khóa API:

In [ ]:
if HAS_KEY:
    interaction = client.interactions.create(
        model=MODEL,
        input="Chào lớp Lập trình xử lý dữ liệu bằng đúng một câu tiếng Việt.",
    )
    print(interaction.output_text)
else:
    print("(DEMO_MODE — bỏ qua lần gọi thử. Kết quả thật sẽ kiểu: 'Chào cả lớp…')")

### Hạn chế của đầu ra tự do

Nếu hỏi "đánh giá này khen gì, chê gì?", mô hình sẽ trả về **văn xuôi** với cách trình bày không ổn định.
Giải pháp là dùng **đầu ra có cấu trúc (structured output)**: khai báo một mẫu trả lời (schema) để mô hình điền theo cấu trúc cố định.

## 4. Đầu ra có cấu trúc: yêu cầu LLM điền biểu mẫu

### 4a. Khai báo schema bằng Pydantic

`Literal` đóng vai trò enum: nhãn ngoài danh sách là **không hợp lệ**. Đây là tầng kiểm tra đầu tiên đối với nhãn do mô hình tạo ra.

In [ ]:
from pydantic import BaseModel
from typing import Literal

Aspect = Literal["location", "cleanliness", "host", "value", "noise", "amenities"]

class ReviewInfo(BaseModel):
    sentiment: Literal["positive", "mixed", "negative"]
    aspects_positive: list[Aspect]   # khía cạnh được KHEN
    aspects_negative: list[Aspect]   # khía cạnh bị CHÊ
    language: str                    # mã ISO 639-1: "en", "es", ... ("und" nếu không rõ)

PROMPT_TEMPLATE = """Bạn là công cụ trích xuất thông tin từ đánh giá về chỗ ở.
Phân tích đánh giá sau và điền đúng schema. Chỉ gán khía cạnh THỰC SỰ được nhắc đến;
đánh giá không có nội dung thì trả sentiment "mixed" và các danh sách rỗng.

Review: {text}"""

print(json.dumps(ReviewInfo.model_json_schema(), indent=2)[:400], "…")

### 4b. Bộ nhớ đệm: không gọi lại kết quả đã có

Lời gọi API được đóng gói trong một hàm có **bộ nhớ đệm (cache)**: kết quả của từng đánh giá được lưu theo `review_id`.
Chạy lại notebook không tạo thêm yêu cầu; khi không có khóa API, notebook đọc cache có sẵn. Đây là cơ chế
`--skip-llm` mà bài tập lớn yêu cầu.

*(Cell dưới dài vì chứa cache đã tính sẵn cho 24 đánh giá. Bạn chỉ cần chạy cell, không cần đọc từng dòng.)*

In [ ]:
# Kết quả LLM đã lưu trong cache cho 24 đánh giá minh hoạ (sinh bằng gemini-3.1-flash-lite, 07/2026)
# Cache được NHÚNG THẲNG vào notebook để chạy ngay trên Colab (không cần key);
# thực tế nên lưu ra file llm_cache.json như slide In[4] để tái dùng giữa các lần chạy.
LLM_CACHE = json.loads("""{
 "1": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"location\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 172,
  "out_tok": 38
 },
 "2": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [\\"location\\"], \\"aspects_negative\\": [\\"cleanliness\\"], \\"language\\": \\"en\\"}",
  "in_tok": 181,
  "out_tok": 44
 },
 "3": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"cleanliness\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"es\\"}",
  "in_tok": 175,
  "out_tok": 40
 },
 "4": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"noise\\"], \\"language\\": \\"es\\"}",
  "in_tok": 174,
  "out_tok": 34
 },
 "5": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [\\"location\\"], \\"aspects_negative\\": [\\"amenities\\"], \\"language\\": \\"pt\\"}",
  "in_tok": 176,
  "out_tok": 42
 },
 "6": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [\\"value\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 170,
  "out_tok": 33
 },
 "7": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"noise\\"], \\"language\\": \\"fr\\"}",
  "in_tok": 178,
  "out_tok": 36
 },
 "8": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"value\\", \\"cleanliness\\", \\"noise\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 179,
  "out_tok": 46
 },
 "9": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"pt\\"}",
  "in_tok": 165,
  "out_tok": 28
 },
 "10": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"cleanliness\\"], \\"language\\": \\"en\\"}",
  "in_tok": 177,
  "out_tok": 35
 },
 "11": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"host\\"], \\"language\\": \\"en\\"}",
  "in_tok": 173,
  "out_tok": 33
 },
 "12": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"cleanliness\\", \\"amenities\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"pt\\"}",
  "in_tok": 180,
  "out_tok": 44
 },
 "13": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"location\\"], \\"language\\": \\"en\\"}",
  "in_tok": 175,
  "out_tok": 36
 },
 "14": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"location\\", \\"cleanliness\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 166,
  "out_tok": 37
 },
 "15": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"es\\"}",
  "in_tok": 176,
  "out_tok": 30
 },
 "16": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"location\\", \\"cleanliness\\"], \\"aspects_negative\\": [], \\"language\\": \\"it\\"}",
  "in_tok": 171,
  "out_tok": 38
 },
 "17": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"noise\\", \\"amenities\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 177,
  "out_tok": 42
 },
 "18": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"cleanliness\\"], \\"language\\": \\"en\\"}",
  "in_tok": 174,
  "out_tok": 34
 },
 "19": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"noise\\"], \\"language\\": \\"en\\"}",
  "in_tok": 172,
  "out_tok": 33
 },
 "20": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [\\"location\\"], \\"aspects_negative\\": [\\"value\\"], \\"language\\": \\"es\\"}",
  "in_tok": 179,
  "out_tok": 43
 },
 "21": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"host\\"], \\"language\\": \\"en\\"}",
  "in_tok": 176,
  "out_tok": 39
 },
 "22": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"amenities\\", \\"location\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 173,
  "out_tok": 36
 },
 "23": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"und\\"}",
  "in_tok": 163,
  "out_tok": 26
 },
 "24": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 160,
  "out_tok": 24
 }
}""")
print(f"Cache có sẵn {len(LLM_CACHE)} kết quả")

In [ ]:
def call_llm_raw(text: str):
    """Gọi Gemini thật, trả về (json_str, in_tok, out_tok)."""
    interaction = client.interactions.create(
        model=MODEL,
        input=PROMPT_TEMPLATE.format(text=text),
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": ReviewInfo.model_json_schema(),
        },
    )
    u = interaction.usage
    return interaction.output_text, u.total_input_tokens, u.total_output_tokens

def extract(review_id: int, text: str, sleep_s: float = 4.0) -> ReviewInfo | None:
    """Trích xuất một đánh giá, ưu tiên cache; thử lại một lần khi gặp lỗi tạm thời."""
    key = str(review_id)
    if key not in LLM_CACHE:                       # chỉ gọi API khi chưa có cache
        if not HAS_KEY:
            return None                            # DEMO_MODE thiếu cache -> không có kết quả
        for attempt in range(2):
            try:
                out, tin, tout = call_llm_raw(text)
                LLM_CACHE[key] = {"output": out, "in_tok": tin, "out_tok": tout}
                time.sleep(sleep_s)                # tuân thủ giới hạn số yêu cầu/phút của gói miễn phí
                break
            except Exception as e:
                print(f"  review {review_id} lỗi lần {attempt+1}: {e}")
                if attempt == 0:
                    time.sleep(20)                 # chỉ chờ khi còn lượt thử
        else:
            return None
    try:
        return ReviewInfo.model_validate_json(LLM_CACHE[key]["output"])
    except Exception as e:                          # schema không khớp -> loại, không cho vào bảng
        print(f"  đánh giá {review_id}: đầu ra không hợp lệ ({e})")
        return None

In [ ]:
# Chạy trích xuất cho 24 đánh giá (dữ liệu minh hoạ đọc từ cache nên chạy ngay)
results = {rid: extract(rid, text) for rid, text in df["comments"].items()}
ok = {rid: r for rid, r in results.items() if r is not None}
print(f"Trích xuất hợp lệ: {len(ok)}/{len(df)}")

### 4c. Chuyển kết quả về DataFrame

Các kết quả trích xuất được ghép thành **bảng nhãn**. Từ đây, dữ liệu có thể được xử lý bằng pandas.

In [ ]:
labels = pd.DataFrame({rid: r.model_dump() for rid, r in ok.items()}).T
labels.index.name = "review_id"
full = df.join(labels, how="left")
full[["comments", "sentiment", "aspects_negative", "language"]].head(8)

In [ ]:
# Câu hỏi mở đầu: khách CHÊ điều gì nhiều nhất?
complaint_counts = (full.explode("aspects_negative")["aspects_negative"]
                    .value_counts())
print(complaint_counts)

fig, ax = plt.subplots(figsize=(6, 3))
complaint_counts.sort_values().plot.barh(ax=ax, color="#1E93AB")
ax.set_title("Khía cạnh bị chê trong 24 đánh giá minh hoạ")
ax.set_xlabel("số đánh giá")
plt.tight_layout()
plt.show()

In [ ]:
# Và bảng cảm xúc × ngôn ngữ — thứ không thể có nếu không trích xuất
pd.crosstab(full["language"], full["sentiment"])

## 5. Đánh giá: LLM cải thiện bao nhiêu so với phương pháp đối chứng?

### 5a. Bộ nhãn chuẩn gán tay (gold labels)

Bộ dữ liệu có **23 đánh giá được đọc và gán nhãn thủ công**; đánh giá 24 rỗng được dành cho mục hậu kiểm.
Trong bài tập lớn, mỗi nhóm phải tự thực hiện quy trình này với **ít nhất 100 đánh giá** và mô tả cách gán nhãn.

In [ ]:
GOLD = {
 "1": {
  "sentiment": "positive",
  "aspects": [
   "location",
   "host"
  ],
  "language": "en"
 },
 "2": {
  "sentiment": "mixed",
  "aspects": [
   "location",
   "cleanliness"
  ],
  "language": "en"
 },
 "3": {
  "sentiment": "positive",
  "aspects": [
   "cleanliness",
   "host"
  ],
  "language": "es"
 },
 "4": {
  "sentiment": "negative",
  "aspects": [
   "noise"
  ],
  "language": "es"
 },
 "5": {
  "sentiment": "mixed",
  "aspects": [
   "location",
   "amenities"
  ],
  "language": "pt"
 },
 "6": {
  "sentiment": "mixed",
  "aspects": [
   "value"
  ],
  "language": "en"
 },
 "7": {
  "sentiment": "mixed",
  "aspects": [
   "noise"
  ],
  "language": "fr"
 },
 "8": {
  "sentiment": "positive",
  "aspects": [
   "value",
   "cleanliness",
   "noise"
  ],
  "language": "en"
 },
 "9": {
  "sentiment": "positive",
  "aspects": [],
  "language": "es"
 },
 "10": {
  "sentiment": "negative",
  "aspects": [
   "cleanliness"
  ],
  "language": "en"
 },
 "11": {
  "sentiment": "negative",
  "aspects": [
   "host"
  ],
  "language": "en"
 },
 "12": {
  "sentiment": "positive",
  "aspects": [
   "cleanliness",
   "amenities",
   "host"
  ],
  "language": "pt"
 },
 "13": {
  "sentiment": "mixed",
  "aspects": [
   "location"
  ],
  "language": "en"
 },
 "14": {
  "sentiment": "positive",
  "aspects": [],
  "language": "en"
 },
 "15": {
  "sentiment": "mixed",
  "aspects": [],
  "language": "es"
 },
 "16": {
  "sentiment": "positive",
  "aspects": [
   "location",
   "cleanliness"
  ],
  "language": "it"
 },
 "17": {
  "sentiment": "positive",
  "aspects": [
   "noise",
   "amenities",
   "host"
  ],
  "language": "en"
 },
 "18": {
  "sentiment": "negative",
  "aspects": [
   "cleanliness"
  ],
  "language": "en"
 },
 "19": {
  "sentiment": "negative",
  "aspects": [
   "noise"
  ],
  "language": "en"
 },
 "20": {
  "sentiment": "mixed",
  "aspects": [
   "location",
   "value"
  ],
  "language": "es"
 },
 "21": {
  "sentiment": "negative",
  "aspects": [
   "host"
  ],
  "language": "en"
 },
 "22": {
  "sentiment": "positive",
  "aspects": [
   "amenities",
   "location"
  ],
  "language": "en"
 },
 "23": {
  "sentiment": "positive",
  "aspects": [],
  "language": "und"
 }
}
GOLD = {int(k): v for k, v in GOLD.items()} if isinstance(list(GOLD)[0], str) else GOLD
print(f"Bộ đánh giá: {len(GOLD)} đánh giá được gán nhãn thủ công")

In [ ]:
# 5b. Chấm điểm: sentiment & language = accuracy; aspects = micro-F1 trên tập nhãn
def as_set(x):                                  # review bị loại -> NaN, coi như tập rỗng
    return set(x) if isinstance(x, list) else set()

def score_flat(pred: dict[int, str], field: str) -> float:
    hit = sum(pred[rid] == g[field] for rid, g in GOLD.items())
    return hit / len(GOLD)

def score_aspects(pred: dict[int, set]) -> float:
    tp = fp = fn = 0
    for rid, g in GOLD.items():
        p, t = set(pred[rid]), set(g["aspects"])
        tp += len(p & t); fp += len(p - t); fn += len(t - p)
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0

llm_sent  = {rid: full.loc[rid, "sentiment"] for rid in GOLD}
llm_lang  = {rid: full.loc[rid, "language"] for rid in GOLD}
llm_asp   = {rid: as_set(full.loc[rid, "aspects_positive"]) | as_set(full.loc[rid, "aspects_negative"])
             for rid in GOLD}
base_sent = {rid: df.loc[rid, "base_sent"] for rid in GOLD}
base_lang = {rid: df.loc[rid, "base_lang"] for rid in GOLD}
base_asp  = {rid: as_set(df.loc[rid, "base_aspects"]) for rid in GOLD}

scoreboard = pd.DataFrame({
    "Baseline": [score_flat(base_sent, "sentiment"), score_aspects(base_asp), score_flat(base_lang, "language")],
    "LLM":      [score_flat(llm_sent, "sentiment"),  score_aspects(llm_asp),  score_flat(llm_lang, "language")],
}, index=["Cảm xúc (accuracy)", "Khía cạnh (micro-F1)", "Ngôn ngữ (accuracy)"]).round(2)
scoreboard

Trên mẫu này, LLM cho kết quả tốt hơn rõ rệt nhưng **vẫn có lỗi**. Ngoài điểm số tổng hợp, cần xem từng trường hợp sai.

In [ ]:
# 5c. Phân tích lỗi: LLM sai ở những đánh giá nào?
errors = []
for rid, g in GOLD.items():
    diff = []
    if llm_sent[rid] != g["sentiment"]:
        diff.append(f"sentiment: {llm_sent[rid]} (đúng: {g['sentiment']})")
    if llm_lang[rid] != g["language"]:
        diff.append(f"language: {llm_lang[rid]} (đúng: {g['language']})")
    extra = llm_asp[rid] - set(g["aspects"])
    if extra:
        diff.append(f"khía cạnh bịa thêm: {sorted(extra)}")
    if diff:
        errors.append({"review_id": rid, "comments": df.loc[rid, "comments"][:60],
                       "lỗi": "; ".join(diff)})
pd.DataFrame(errors)

Ba trường hợp trên đại diện cho ba **kiểu lỗi thường gặp** của LLM:

1. **Diễn giải ranh giới nhãn khác người gán** (đánh giá 2: `negative` thay vì `mixed`) — ranh giới nhãn
   phải được định nghĩa rõ trong hướng dẫn gán *và* trong prompt.
2. **Nhầm ngôn ngữ gần nhau** trên câu ngắn (đánh giá 9: es → pt).
3. **Tạo thêm khía cạnh không có trong văn bản** (đánh giá 14) — đây là lỗi bịa thông tin (hallucination),
   dễ bị bỏ qua khi chỉ đọc lướt đầu ra.

Sau khi sửa prompt, cần **đo lại trên đúng bộ nhãn chuẩn**, tương tự việc chạy lại test sau khi sửa mã nguồn.

In [ ]:
# 5d. Hậu kiểm (post-check) tự động: bắt các kết quả "đáng ngờ"
suspicious = []
for rid, r in ok.items():
    text = df.loc[rid, "comments"]
    n_aspects = len(r.aspects_positive) + len(r.aspects_negative)
    if len(text.strip()) == 0 and (n_aspects > 0 or r.sentiment != "mixed"):
        suspicious.append((rid, "đánh giá rỗng mà vẫn có nhãn!"))
    if len(text.split()) <= 4 and n_aspects >= 2:
        suspicious.append((rid, f"đánh giá có {len(text.split())} từ nhưng được gán {n_aspects} khía cạnh"))
for rid, why in suspicious:
    print(f"⚠️ đánh giá {rid}: {why} — {df.loc[rid, 'comments']!r}")

Hậu kiểm phát hiện ngay đánh giá 24 rỗng: prompt yêu cầu *"đánh giá không có nội dung thì trả mixed + danh sách rỗng"*
nhưng mô hình vẫn trả `positive`. **Schema chặn được nhãn lạ nhưng không chặn được nhãn hợp lệ về hình thức mà
không có căn cứ**. Vì vậy, đầu ra LLM cần thêm tầng hậu kiểm bằng quy tắc, theo tư duy QA ở bài 10.

In [ ]:
# 5e. Chi phí: cộng token đã dùng và quy ra tiền nếu chạy quy mô thật
tin = sum(v["in_tok"] for v in LLM_CACHE.values())
tout = sum(v["out_tok"] for v in LLM_CACHE.values())
print(f"24 đánh giá minh hoạ: {tin:,} token vào + {tout:,} token ra")

# Giá gemini-3.1-flash-lite (07/2026): $0.25 / 1M token vào, $1.50 / 1M token ra
# Gói miễn phí flash-lite tại thời điểm soạn bài: ~1.000 yêu cầu/ngày (kiểm tra lại trong AI Studio)
per_review_in, per_review_out = tin / 24, tout / 24
for n in [2_000, 100_000, 690_000]:
    cost = (n * per_review_in * 0.25 + n * per_review_out * 1.50) / 1e6
    days_free = n / 1_000
    print(f"{n:>9,} đánh giá  ≈  ${cost:>6,.2f} trả phí   |   ~{days_free:,.0f} ngày nếu dùng gói miễn phí")

Kết luận thực tế: nếu **chọn mẫu phù hợp** gồm vài nghìn đánh giá phục vụ đúng câu hỏi và dùng **cache**,
gói miễn phí có thể đáp ứng bài tập lớn. Việc xử lý toàn bộ dữ liệu cần được cân nhắc bằng chi phí đã ước tính.

## 6. Bài tập tại lớp

Làm ngay tại chỗ, 15–20 phút. Sửa trực tiếp các cell dưới.

### Bài 1 — Mở rộng danh mục khía cạnh

Đánh giá 15 nhắc đến **an toàn** (*"el barrio se siente inseguro"*) nhưng enum chưa có nhãn
`safety`, nên thông tin này không được ghi nhận. Hãy:

1. Trở lại **cell 16**, thêm `"safety"` vào `Aspect`, rồi **chạy lại cell 16** để `ReviewInfo` chấp nhận nhãn mới.
2. Thêm từ khoá `safety` cho phương pháp đối chứng (`ASPECT_KEYWORDS`).
3. Chạy lại bước trích xuất cho **riêng đánh giá 15** sau khi xoá cache bằng `LLM_CACHE.pop("15", None)`.
   Nếu **không có khóa API**, hãy tự viết JSON đúng schema mới, có `safety`, rồi lưu kết quả vào cache.
4. Cập nhật nhãn chuẩn của đánh giá 15, rồi **chạy lại cell 20 → 22 → 27**. Điểm số thay đổi thế nào?

In [ ]:
# Bài 1 — làm theo 4 bước ở trên. Bước 1 (sửa Aspect ở cell 16 + chạy lại cell 16) phải xong TRƯỚC:
assert "safety" in Aspect.__args__, "Chưa thêm 'safety' vào Aspect ở cell 16 — sửa rồi chạy lại cell 16."

# TODO 2: thêm từ khoá cho phương pháp đối chứng (nên bao gồm cả tiếng Tây Ban Nha)
ASPECT_KEYWORDS["safety"] = ...        # vd r"inseguro|unsafe|safety|crime|robo"

# TODO 3: xoá cache của đánh giá 15; nếu KHÔNG có khóa API, tự viết JSON đúng schema mới:
LLM_CACHE.pop("15", None)
# LLM_CACHE["15"] = {"output": '{"sentiment": "mixed", "aspects_positive": [],'
#                              ' "aspects_negative": ["safety"], "language": "es"}',
#                    "in_tok": 0, "out_tok": 0}

# TODO 4: cập nhật nhãn chuẩn của đánh giá 15 (thêm 'safety')
# GOLD[15]["aspects"] = [...]

print("Xong 4 bước? -> chạy lại cell 20 -> 22 -> 27 để xem điểm đổi thế nào.")

### Bài 2 — Viết thêm một quy tắc hậu kiểm

Mục 5d mới có hai quy tắc. Hãy thêm quy tắc thứ ba: **mỗi khía cạnh được gán phải có bằng chứng trong văn
bản**. Với mỗi khía cạnh mà LLM trả về, nếu văn bản không khớp từ khoá nào tương ứng trong
`ASPECT_KEYWORDS`, hãy in cảnh báo. Quy tắc này có phát hiện lỗi của đánh giá 14 không? Khi nào nó có thể
cảnh báo nhầm? Gợi ý: đánh giá được viết bằng ngôn ngữ mà bộ từ khoá chưa bao phủ.

In [ ]:
# TODO Bài 2 — hoàn thiện hàm dưới:
def check_aspect_evidence(rid: int, r: ReviewInfo) -> list[str]:
    text = df.loc[rid, "comments"]
    warnings = []
    for aspect in set(r.aspects_positive) | set(r.aspects_negative):
        pat = ASPECT_KEYWORDS.get(aspect, "")
        # TODO: nếu có pat mà text KHÔNG khớp (re.search(pat, text, flags=re.I) is None)
        #       -> thêm một cảnh báo vào warnings
        ...
    return warnings

for rid, r in ok.items():
    for w in check_aspect_evidence(rid, r):
        print(f"⚠️ review {rid}: {w} — {df.loc[rid, 'comments'][:50]!r}")

### Bài 3 — Sửa prompt, đo lại

Lỗi ở đánh giá 2 (`negative` thay vì `mixed`) xuất phát từ ranh giới nhãn chưa rõ. Hãy sửa
`PROMPT_TEMPLATE` bằng cách thêm định nghĩa *"mixed = có cả ý khen lẫn ý chê"*. Nếu có khóa API,
hãy xoá cache của đánh giá 2 và chạy lại. Điểm cảm xúc có tăng không? Mỗi phiên bản prompt cần được
lưu lại cùng điểm số tương ứng để tránh thử nghiệm thiếu hệ thống.

In [ ]:
# TODO Bài 3 — viết prompt v2 của bạn:
# Gợi ý: PROMPT_TEMPLATE.replace("điền đúng schema.", "điền đúng schema. Quy ước: ...")
PROMPT_V2 = ...

# print(PROMPT_V2)          # bỏ dấu # khi đã điền

## 7. Bài tập về nhà

Lấy **200 đánh giá thật** của một thành phố trong đề bài tập lớn và chạy toàn bộ pipeline của bài này:

1. Tải `reviews.csv.gz` (URL mẫu trong cell dưới), lấy 200 đánh giá mới nhất có độ dài ≥ 30 ký tự.
2. Chạy phương pháp đối chứng và LLM; lưu ý hạn mức khoảng 200 yêu cầu, dùng `time.sleep` và lưu cache ra file JSON.
3. Tự gán nhãn cho 20 đánh giá làm bộ nhãn chuẩn, rồi tính bảng điểm như mục 5.
4. Viết 5 câu nhận xét: LLM có phù hợp với bảng `reviews` của thành phố này không? Chi phí ước tính
   nếu chạy trên toàn bộ đánh giá?

Nộp notebook và file cache JSON để người chấm có thể chạy lại **mà không cần khóa API**.

In [ ]:
RUN_CHALLENGE = False   # đổi thành True khi làm ở nhà

if RUN_CHALLENGE:
    URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
           "2026-06-29/data/reviews.csv.gz")          # đổi theo thành phố nhóm bạn
    rv = pd.read_csv(URL)
    rv = rv[rv["comments"].str.len() >= 30]
    sample = rv.sort_values("date").tail(200)
    print(sample.shape)

---

## Tóm tắt bài học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Phi cấu trúc → **đổi dạng thành bảng** | Dùng lại các công cụ pandas đã học |
| **Phương pháp đối chứng trước, LLM sau** | Cần mốc so sánh để lượng hoá mức cải thiện |
| Schema + enum + giới hạn tốc độ + **cache** | Pipeline LLM chạy lại được mà không lãng phí hạn mức |
| **Bộ nhãn tay** + hậu kiểm + đếm token | Chất lượng và chi phí đều phải *đo*, không *đoán* |

📌 Các kỹ năng này được dùng trực tiếp trong **hợp phần LLM của bài tập lớn**. Xem đề để biết yêu cầu đầy đủ
(phương pháp đối chứng, ≥100 nhãn tay, báo cáo chi phí, `--skip-llm`).